In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!cp -r /kaggle/input/datasets/xauskeleton/modell/Prune_QT_VITs/* /kaggle/working/
%cd /kaggle/working
!ls



In [ ]:
!pip install -q datasets timm


In [ ]:
!python apb_fimaq/qat.py --debug --out-dir /kaggle/working/qat


In [ ]:
%%bash
cd /kaggle/working
# ============================================================================
# KẾT QUẢ CUỐI — combined prune→quant @ 30ep (init pruned g05_fisher, magnitude+DPLR)
# Mỗi lần chạy: BỎ COMMENT đúng 1 dòng dưới (3 dòng kia để #), rồi Save & Run All.
# Tất cả ~8-10h → lọt cap 12h, không lo cắt.
# ----------------------------------------------------------------------------
OUT=/kaggle/working/checkpoints/final_pruned_br0.95_act2_30ep;  ARGS="--binary-ratio 0.95 --act-bits 2"
# OUT=/kaggle/working/checkpoints/final_pruned_br0.99_act2_30ep; ARGS="--binary-ratio 0.99 --act-bits 2"
# OUT=/kaggle/working/checkpoints/final_pruned_br0.95_act1_30ep; ARGS="--binary-ratio 0.95 --act-bits 1"
# OUT=/kaggle/working/checkpoints/final_pruned_br0.99_act1_30ep; ARGS="--binary-ratio 0.99 --act-bits 1"
# ============================================================================

python apb_fimaq/qat.py --dataset cifar100 --init-model ckpt/best_pruned_g05_fisher.pt \
  --apb-scope full --partition magnitude --use-dplr-loss --dplr-lambda 3000 \
  --epochs 30 --fim-batches 10 --lr 1e-4 --batch-size 32 --num-workers 4 --seed 3407 \
  $ARGS --out-dir "$OUT"
